In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
# ===================== CELL 1 — Setup, Hugging Face login, private repos =====================
# Notebook 1 = DATA PREP. Settings: Accelerator = None (CPU, saves GPU quota), Internet = On.
!pip install -q ijson

import os
os.environ["HF_HOME"] = "/tmp/hf_home"          # keep HF cache on the big disk, not /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "true"

import sys, json, time, shutil, random, math, gc
import numpy as np
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, HfApi
import huggingface_hub, tokenizers

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)
api = HfApi(token=HF_TOKEN)

me = api.whoami()
HF_USER = me["name"]
token_role = me.get("auth", {}).get("accessToken", {}).get("role", "unknown")
print(f"HF user: {HF_USER} | token role: {token_role}")

SRC_REPO  = "Exploration-Lab/CS-781-Capstone"     # official competition data (read only)
DATA_REPO = f"{HF_USER}/cs781-p1-data"            # OUR private repo: tokenizer + tokenized corpus + dev set
CKPT_REPO = f"{HF_USER}/cs781-p1-ckpt"            # OUR private repo: training checkpoints
api.create_repo(DATA_REPO, repo_type="dataset", private=True, exist_ok=True)
api.create_repo(CKPT_REPO, repo_type="model",   private=True, exist_ok=True)
print("Private repos ready:", DATA_REPO, "|", CKPT_REPO)

WORK    = "/tmp/cs781"                 # large scratch disk
RAW_DIR = f"{WORK}/raw"                # downloaded official json files
OUT_DIR = f"{WORK}/prepared"           # tokenizer + .bin files we build
os.makedirs(RAW_DIR, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

for p in ["/tmp", "/kaggle/working"]:
    du = shutil.disk_usage(p)
    print(f"Disk {p:16s} free {du.free/1e9:6.1f} GB / total {du.total/1e9:6.1f} GB")
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
print(f"RAM {ram:.1f} GB | CPUs {os.cpu_count()} | python {sys.version.split()[0]} | "
      f"huggingface_hub {huggingface_hub.__version__} | tokenizers {tokenizers.__version__}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF user: Abhayk24 | token role: fineGrained
Private repos ready: Abhayk24/cs781-p1-data | Abhayk24/cs781-p1-ckpt
Disk /tmp             free 1148.8 GB / total 8758.4 GB
Disk /kaggle/working  free   20.9 GB / total   21.0 GB
RAM 33.7 GB | CPUs 4 | python 3.13.15 | huggingface_hub 1.29.0 | tokenizers 0.23.1


In [3]:
# ===================== CELL 2 — List the official Phase-1 files and their sizes =====================
top = list(api.list_repo_tree(SRC_REPO, repo_type="dataset"))
print("Top level of the official repo:")
for x in top:
    print("  ", x.path, f"{x.size/1e9:.2f} GB" if getattr(x, "size", None) else "(folder)")

P1_DIR = next(x.path for x in top if x.path.lower().startswith("phase_1"))
p1_files = [x for x in api.list_repo_tree(SRC_REPO, repo_type="dataset", path_in_repo=P1_DIR, recursive=True)
            if getattr(x, "size", None) is not None]
print(f"\nFiles under {P1_DIR}:")
for x in p1_files:
    print(f"   {x.path:70s} {x.size/1e9:8.3f} GB")

TEST_PATH = next(x.path for x in p1_files if "/test" in x.path.lower() and x.path.endswith(".json"))
TRAIN_PATHS = {}
for x in p1_files:
    name = x.path.split("/")[-1].lower()
    for src in ["acts", "high_courts", "supreme_court", "tribunals"]:
        if name == f"train_{src}.json":
            TRAIN_PATHS[src] = x.path
TRAIN_SIZES = {s: next(x.size for x in p1_files if x.path == p) for s, p in TRAIN_PATHS.items()}
print("\nTEST_PATH  :", TEST_PATH)
print("TRAIN_PATHS:", json.dumps(TRAIN_PATHS, indent=2))
assert len(TRAIN_PATHS) == 4, "Did not find all 4 train files - paste this output to Claude."
print(f"Total train JSON size: {sum(TRAIN_SIZES.values())/1e9:.2f} GB")

Top level of the official repo:
   Phase_1_Pretraining (folder)
   Phase_2_LNER (folder)
   Phase_3_RR (folder)
   Phase_4_CJPE (folder)
   Phase_5_LSI (folder)
   Phase_6_Summ (folder)
   Phase_7_PCR (folder)
   .gitattributes 0.00 GB
   README.md 0.00 GB

Files under Phase_1_Pretraining:
   Phase_1_Pretraining/Test/test.json                                        0.003 GB
   Phase_1_Pretraining/Train/train_acts.json                                 0.729 GB
   Phase_1_Pretraining/Train/train_high_courts.json                         10.672 GB
   Phase_1_Pretraining/Train/train_supreme_court.json                        1.323 GB
   Phase_1_Pretraining/Train/train_tribunals.json                            6.484 GB

TEST_PATH  : Phase_1_Pretraining/Test/test.json
TRAIN_PATHS: {
  "acts": "Phase_1_Pretraining/Train/train_acts.json",
  "high_courts": "Phase_1_Pretraining/Train/train_high_courts.json",
  "supreme_court": "Phase_1_Pretraining/Train/train_supreme_court.json",
  "tribunals": "Ph

In [5]:
# ===================== CELL 3 — Download the official TEST prompts and study them =====================
from huggingface_hub import hf_hub_download

test_local = hf_hub_download(SRC_REPO, TEST_PATH, repo_type="dataset", token=HF_TOKEN, local_dir=RAW_DIR)
with open(test_local, encoding="utf-8") as f:
    test_data = json.load(f)

print("Top-level type:", type(test_data).__name__, "| examples:", len(test_data))
if isinstance(test_data, dict):
    test_ids, test_recs = list(test_data.keys()), list(test_data.values())
else:
    test_ids, test_recs = [str(i) for i in range(len(test_data))], list(test_data)
    print("NOTE: test file is a LIST. First element:", str(test_data[0])[:300])
print("First ids:", test_ids[:5], "... last ids:", test_ids[-3:])
print("Record keys:", sorted({k for r in test_recs if isinstance(r, dict) for k in r}))

test_texts = [r["text"] if isinstance(r, dict) else r for r in test_recs]
wc = np.array([len(t.split()) for t in test_texts])
print(f"Words per prompt: min {wc.min()} | mean {wc.mean():.1f} | max {wc.max()} | exactly 512: {(wc == 512).sum()}")
print("Prompts containing a newline     :", sum("\n" in t for t in test_texts))
print("Prompts containing a double space:", sum("  " in t for t in test_texts))
print("Prompts with leading/trailing ws :", sum(t != t.strip() for t in test_texts))
chars = sum(len(t) for t in test_texts)
print(f"Non-ASCII character share: {sum(ord(c) > 127 for t in test_texts for c in t) / chars:.4%}")
for i in [0, len(test_texts) // 2]:
    t = test_texts[i]
    print(f"\n----- test id {test_ids[i]} -----\nSTART: {t[:500]!r}\n...\nEND  : {t[-300:]!r}")

test.json:   0%|          | 0.00/3.03M [00:00<?, ?B/s]

Top-level type: dict | examples: 1000
First ids: ['1', '2', '3', '4', '5'] ... last ids: ['998', '999', '1000']
Record keys: ['text']
Words per prompt: min 512 | mean 512.0 | max 512 | exactly 512: 1000
Prompts containing a newline     : 0
Prompts containing a double space: 0
Prompts with leading/trailing ws : 0
Non-ASCII character share: 0.0000%

----- test id 1 -----
START: 'The issue involved in these petitions is as to whether the acquisition proceedings can be challenged at a belated stage. The issue is no more res integra as the issue has been considered by this Court time and again. 6. When a person challenges Section 4 Notification on any ground, it should be challenged within a reasonable period, and if the acquisition is challenged at a belated stage, the petition deserves to be dismissed only on this count. (Vide Hari Singh & Ors. Vs. State of U.P., AIR 198'
...
END  : 'dings need not be set at naught if the party has not approached the Court within reasonable time, as judic

In [6]:
# ===================== CELL 4 — Download the 4 train files + stream-peek their structure =====================
import ijson

def detect_layout(path):
    """'list' = [{...},...]   'dict' = {"id": {...}}   'jsonl' = one JSON object per line."""
    with open(path, "rb") as f:
        ev = ijson.parse(f, multiple_values=True)
        e1 = next(ev)
        if e1[1] == "start_array":
            return "list"
        if e1[1] != "start_map":
            raise ValueError(f"Unexpected JSON start: {e1}")
        next(ev); e3 = next(ev)
        if e3[1] == "start_map":
            return "dict"
    with open(path, "rb") as f:
        line = f.readline(64 * 2**20); more = f.read(64).strip()
    try:
        json.loads(line); return "jsonl" if more else "dict"
    except Exception:
        return "dict"

def iter_records(path):
    """Stream (id, text, raw_record) without loading the whole file into RAM."""
    layout = detect_layout(path)
    with open(path, "rb") as f:
        if layout == "list":   it = ((None, r) for r in ijson.items(f, "item"))
        elif layout == "dict": it = ijson.kvitems(f, "")
        else:                  it = ((None, r) for r in ijson.items(f, "", multiple_values=True))
        for i, (k, r) in enumerate(it):
            text = r if isinstance(r, str) else r.get("text")
            yield (str(k) if k is not None else str(i)), text, r

print("ijson backend:", ijson.backend, "(want yajl2_c for speed)")
need = sum(TRAIN_SIZES.values())
free = shutil.disk_usage(WORK).free
print(f"Need {need/1e9:.1f} GB, free {free/1e9:.1f} GB")
assert free > need + 15e9, "Not enough disk - paste this output to Claude."

TRAIN_LOCAL = {}
for src, path in TRAIN_PATHS.items():
    t0 = time.time()
    TRAIN_LOCAL[src] = hf_hub_download(SRC_REPO, path, repo_type="dataset", token=HF_TOKEN, local_dir=RAW_DIR)
    print(f"downloaded {src:14s} {os.path.getsize(TRAIN_LOCAL[src])/1e9:6.2f} GB in {time.time()-t0:5.0f}s")

for src, path in TRAIN_LOCAL.items():
    print(f"\n================ {src} | layout = {detect_layout(path)} ================")
    n, words, nl_docs, nl_count, nbytes, empty = 0, [], 0, 0, 0, 0
    for k, text, rec in iter_records(path):
        if n < 2:
            keys = list(rec.keys()) if isinstance(rec, dict) else type(rec).__name__
            print(f"  id={k!r} keys={keys}\n  text[:300]={str(text)[:300]!r}")
        if not text or not str(text).strip():
            empty += 1
        else:
            words.append(len(text.split())); nl_docs += "\n" in text
            nl_count += text.count("\n"); nbytes += len(text.encode("utf-8"))
        n += 1
        if n >= 3000:
            break
    w = np.array(words)
    print(f"  first {n} docs: empty {empty} | words/doc median {np.median(w):.0f} mean {w.mean():.0f} max {w.max()} "
          f"| docs >= 640 words: {(w >= 640).mean():.1%} | docs with newline: {nl_docs/len(w):.1%} "
          f"| newlines per 1k words: {1000*nl_count/w.sum():.1f}")
    est_words = os.path.getsize(path) * (w.sum() / nbytes)
    print(f"  rough estimate of total words in file: {est_words/1e9:.2f} B")

ijson backend: yajl2_c (want yajl2_c for speed)
Need 19.2 GB, free 1148.8 GB


Phase_1_Pretraining/Train/train_acts.jso(…): reconstructing file:   0%|          |  0.00B /  729MB            

Phase_1_Pretraining/Train/train_acts.jso(…): downloading bytes:           |  0.00B            

downloaded acts             0.73 GB in     3s


Phase_1_Pretraining/Train/train_high_cou(…): reconstructing file:   0%|          |  0.00B / 10.7GB            

Phase_1_Pretraining/Train/train_high_cou(…): downloading bytes:           |  0.00B            

... (3 outputs left out)

downloaded supreme_court    1.32 GB in    18s


Phase_1_Pretraining/Train/train_tribunal(…): reconstructing file:   0%|          |  0.00B / 6.48GB            

Phase_1_Pretraining/Train/train_tribunal(…): downloading bytes:           |  0.00B            

downloaded tribunals        6.48 GB in    31s

================ acts | layout = list ================
  id='0' keys=['title', 'category', 'year', 'url', 'text', 'word_count', 'text_length_chars']
  text[:300]='Bangalore City Planning Area Zonal Regulations (Amendment and Validation) Act, 1996 (Karnataka Act No. 2 of 1996) Statements of objects and Reasons - Act 2 of 1996. - The Bangalore City Planning Area Zonal Regulations 1972, which was in vogue till 1984, provided for construction of five floors inclu'
  id='1' keys=['title', 'category', 'year', 'url', 'text', 'word_count', 'text_length_chars']
  text[:300]='The Bangalore Palace (Acquisition And Transfer) Act, 1996 [18th November, 1996] Karnataka Act No. 18 of 1996 An Act to provide for the acquisition and transfer of the Bangalore Palace and open space around it in the public interest and for its preservation and for matters connected therewith. WHEREA'
  first 3000 docs: empty 0 | words/doc median 2544 mean 6305 max 277710 | docs

In [7]:
# ===================== CELL 5 - Normalise text, split train/val/dev, subsample the big sources =====================
# What Cells 3-4 showed: test prompts are exactly 512 words, single-spaced, NO newlines, pure ASCII, and they are
# windows taken from the MIDDLE of Supreme-Court-style judgments. So we (a) normalise every training document the
# same way, (b) hold out Supreme Court documents to build a local dev set that mimics the test.
import unicodedata, hashlib

TEXT_DIR = f"{OUT_DIR}/text"; os.makedirs(TEXT_DIR, exist_ok=True)
SEED = 1234
PROMPT_WORDS, CONT_WORDS = 512, 128
MIN_WORDS = 50                     # drop near-empty docs ("Heard. Dismissed.")
VAL_CAP_WORDS = 3_000_000          # per source, for validation loss

# dev = held-out docs for generation/BERTScore; val = held-out docs for loss; keep = fraction of remaining docs
# used for training; budget = max training words taken from that source (None = take everything).
PLAN = {
    "supreme_court": dict(dev=0.010, val=0.010, keep=1.00, budget=None),
    "acts":          dict(dev=0.000, val=0.010, keep=1.00, budget=None),
    "tribunals":     dict(dev=0.000, val=0.003, keep=0.35, budget=330_000_000),
    "high_courts":   dict(dev=0.000, val=0.002, keep=0.21, budget=330_000_000),
}

PUNCT = str.maketrans({"\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'", "\u2032": "'",
                       "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u2033": '"',
                       "\u2010": "-", "\u2011": "-", "\u2012": "-", "\u2013": "-", "\u2014": "-",
                       "\u2015": "-", "\u2212": "-", "\u00a0": " ", "\u2022": "*", "\u00b7": "."})

def normalize(t):
    """NFKC -> ASCII punctuation -> drop remaining non-ASCII -> collapse all whitespace to single spaces."""
    t = unicodedata.normalize("NFKC", t).translate(PUNCT)
    a = t.encode("ascii", "ignore").decode("ascii")
    return " ".join(a.split()), len(t) - len(a)

def u01(key):
    """Deterministic uniform number in [0,1) from a string (same split every time we rerun)."""
    return int(hashlib.md5(f"{SEED}|{key}".encode()).hexdigest()[:12], 16) / 16**12

dev_prompts, dev_truth, STATS = {}, {}, {}
for src in ["supreme_court", "acts", "tribunals", "high_courts"]:
    cfg, t0 = PLAN[src], time.time()
    s = dict(docs=0, too_short=0, not_sampled=0, train_docs=0, train_words=0, val_docs=0, val_words=0,
             dev_docs=0, chars=0, nonascii_removed=0)
    f_train = open(f"{TEXT_DIR}/{src}_train.txt", "w", encoding="utf-8")
    f_val   = open(f"{TEXT_DIR}/{src}_val.txt",   "w", encoding="utf-8")
    for idx, (k, text, rec) in enumerate(iter_records(TRAIN_LOCAL[src])):
        s["docs"] += 1
        u = u01(f"{src}|{idx}")
        if u < cfg["dev"]:
            split = "dev"
        elif u < cfg["dev"] + cfg["val"]:
            split = "val"
            if s["val_words"] >= VAL_CAP_WORDS:
                continue
        else:
            split = "train"
            if (cfg["budget"] and s["train_words"] >= cfg["budget"]) or u01(f"{src}|keep|{idx}") >= cfg["keep"]:
                s["not_sampled"] += 1
                continue
        if not text:
            s["too_short"] += 1; continue
        norm, removed = normalize(str(text))
        s["chars"] += len(norm); s["nonascii_removed"] += removed
        words = norm.split(" ") if norm else []
        if len(words) < MIN_WORDS:
            s["too_short"] += 1; continue
        if split == "dev":
            if len(words) >= PROMPT_WORDS + CONT_WORDS:          # one test-like window per dev document
                st = random.Random(f"{SEED}|{src}|{idx}").randint(0, len(words) - PROMPT_WORDS - CONT_WORDS)
                ex_id = str(len(dev_prompts) + 1)
                dev_prompts[ex_id] = {"text": " ".join(words[st:st + PROMPT_WORDS])}
                dev_truth[ex_id]   = {"text": " ".join(words[st + PROMPT_WORDS: st + PROMPT_WORDS + CONT_WORDS])}
                s["dev_docs"] += 1
        elif split == "val":
            f_val.write(norm + "\n"); s["val_docs"] += 1; s["val_words"] += len(words)
        else:
            f_train.write(norm + "\n"); s["train_docs"] += 1; s["train_words"] += len(words)
        if cfg["budget"] and s["train_words"] >= cfg["budget"] and s["val_words"] >= VAL_CAP_WORDS:
            print(f"   {src}: budget reached after {s['docs']:,} docs - stopping early"); break
        if s["docs"] % 50_000 == 0:
            print(f"   {src}: {s['docs']:,} docs scanned, {s['train_words']/1e6:,.0f}M train words, {time.time()-t0:,.0f}s")
    f_train.close(); f_val.close()
    s["nonascii_per_1k_chars"] = round(1000 * s["nonascii_removed"] / max(1, s["chars"] + s["nonascii_removed"]), 3)
    s["seconds"] = round(time.time() - t0)
    STATS[src] = s
    print(f"{src:14s} docs {s['docs']:>9,} | train {s['train_docs']:>8,} docs {s['train_words']/1e6:8.1f}M words | "
          f"val {s['val_docs']:>6,} docs {s['val_words']/1e6:5.2f}M words | dev {s['dev_docs']:>4} | "
          f"short {s['too_short']:,} | not sampled {s['not_sampled']:,} | non-ASCII/1k {s['nonascii_per_1k_chars']} "
          f"| {s['seconds']}s")

with open(f"{OUT_DIR}/dev.json", "w", encoding="utf-8") as f:
    json.dump(dev_prompts, f, ensure_ascii=False)
with open(f"{OUT_DIR}/dev_truth.json", "w", encoding="utf-8") as f:
    json.dump(dev_truth, f, ensure_ascii=False)
with open(f"{OUT_DIR}/prep_stats.json", "w") as f:
    json.dump({"plan": PLAN, "stats": STATS, "seed": SEED, "min_words": MIN_WORDS}, f, indent=2)
assert all(len(v["text"].split()) == 512 for v in dev_prompts.values())
assert all(len(v["text"].split()) == 128 for v in dev_truth.values())
print(f"\nDev examples (Supreme Court, held out from training): {len(dev_prompts)}")
print("Example dev prompt end :", dev_prompts["1"]["text"][-200:])
print("Example dev reference  :", dev_truth["1"]["text"][:200])

   supreme_court: 50,000 docs scanned, 186M train words, 96s
supreme_court  docs    55,549 | train   54,333 docs    206.2M words | val    555 docs  2.06M words | dev  479 | short 108 | not sampled 0 | non-ASCII/1k 0.144 | 107s
acts           docs    18,826 | train   18,609 docs    118.3M words | val    208 docs  1.38M words | dev    0 | short 9 | not sampled 0 | non-ASCII/1k 0.081 | 75s
   tribunals: 400,000 docs scanned, 138M train words, 99s
   tribunals: 450,000 docs scanned, 160M train words, 109s
   tribunals: 500,000 docs scanned, 181M train words, 122s
   tribunals: 850,000 docs scanned, 329M train words, 224s
   tribunals: budget reached after 850,659 docs - stopping early
tribunals      docs   850,659 | train  294,980 docs    330.0M words | val  2,477 docs  3.00M words | dev    0 | short 1,122 | not sampled 552,023 | non-ASCII/1k 16.929 | 224s
   high_courts: 100,000 docs scanned, 9M train words, 5s
   high_courts: 350,000 docs scanned, 32M train words, 18s
   high_courts: 550

In [8]:
# ===================== CELL 6 - Train our own byte-level BPE tokenizer on legal text =====================
# Same algorithm as the course BPE (byte-level, merge most frequent pair), but the Rust `tokenizers` implementation,
# which handles hundreds of MB in minutes. 32,768 = 2^15 entries: fits uint16 storage and is a multiple of 64 (GPU-friendly).
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

VOCAB_SIZE = 32768
SPECIALS = ["<|endoftext|>", "<|pad|>", "<|bos|>", "<|sep|>", "<|mask|>", "<|unk|>"] + \
           [f"<|reserved_{i}|>" for i in range(10)]          # reserved ids 6..15 for Phases 2-7 (tags, separators)
SAMPLE_P = {"supreme_court": 0.25, "acts": 0.15, "tribunals": 0.06, "high_courts": 0.06}   # weighted to the test domain

def sample_batches(batch_size=1000):
    n_docs = n_chars = 0
    for src, p in SAMPLE_P.items():
        r, batch = random.Random(f"tok|{src}"), []
        with open(f"{TEXT_DIR}/{src}_train.txt", encoding="utf-8") as f:
            for line in f:
                if r.random() < p:
                    line = line.rstrip("\n"); batch.append(line); n_docs += 1; n_chars += len(line)
                    if len(batch) == batch_size:
                        yield batch; batch = []
        if batch:
            yield batch
    print(f"   tokenizer trained on {n_docs:,} docs / {n_chars/1e6:,.0f}M characters")

t0 = time.time()
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True)
tok.decoder = decoders.ByteLevel()
trainer = trainers.BpeTrainer(vocab_size=VOCAB_SIZE, min_frequency=2, special_tokens=SPECIALS,
                              initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=False)
tok.train_from_iterator(sample_batches(), trainer=trainer)
tok.save(f"{OUT_DIR}/tokenizer.json")
print(f"Tokenizer trained in {time.time()-t0:.0f}s | vocab {tok.get_vocab_size()}")

assert tok.get_vocab_size() == VOCAB_SIZE
for i, sp in enumerate(SPECIALS):
    assert tok.token_to_id(sp) == i, sp
EOS_ID = tok.token_to_id("<|endoftext|>")

# Round-trip check on the official test prompts: decode(encode(x)) must give back x exactly.
bad = sum(tok.decode(tok.encode(t).ids) != t for t in test_texts)
print("Round-trip mismatches on the test prompts:", bad)
assert bad == 0

# Fertility = tokens per word (lower = more text fits in the context, cheaper training and generation).
ours = float(np.mean([len(tok.encode(t).ids) / 512 for t in test_texts]))
try:
    gpt2_tok = Tokenizer.from_pretrained("openai-community/gpt2")
    gpt2 = float(np.mean([len(gpt2_tok.encode(t).ids) / 512 for t in test_texts]))
except Exception as e:
    gpt2 = float("nan"); print("(GPT-2 tokenizer comparison skipped:", e, ")")
print(f"Tokens per word on test prompts: ours {ours:.3f} | GPT-2 {gpt2:.3f}  "
      f"-> a 512-word prompt is ~{ours*512:.0f} tokens with ours vs ~{gpt2*512:.0f} with GPT-2")
ex = test_texts[0][:160]
print("Example:", [tok.id_to_token(i) for i in tok.encode(ex).ids][:40])

   tokenizer trained on 64,071 docs / 648M characters
Tokenizer trained in 144s | vocab 32768
Round-trip mismatches on the test prompts: 0


tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Tokens per word on test prompts: ours 1.197 | GPT-2 1.240  -> a 512-word prompt is ~613 tokens with ours vs ~635 with GPT-2
Example: ['The', 'Ġissue', 'Ġinvolved', 'Ġin', 'Ġthese', 'Ġpetitions', 'Ġis', 'Ġas', 'Ġto', 'Ġwhether', 'Ġthe', 'Ġacquisition', 'Ġproceedings', 'Ġcan', 'Ġbe', 'Ġchallenged', 'Ġat', 'Ġa', 'Ġbelated', 'Ġstage', '.', 'ĠThe', 'Ġissue', 'Ġis', 'Ġno', 'Ġmore', 'Ġres', 'Ġintegra', 'Ġas', 'Ġthe']


In [9]:
# ===================== CELL 7 - Tokenize every split into compact uint16 .bin files =====================
# One document per line -> token ids + <|endoftext|> separator, appended to a flat uint16 file (2 bytes/token).
# The trainer memory-maps these files, so RAM use stays tiny no matter how many tokens we have.
BIN_DIR = f"{OUT_DIR}/bin"; os.makedirs(BIN_DIR, exist_ok=True)
assert VOCAB_SIZE <= 65535

def tokenize_file(src, split, batch_size=512):
    inp, out = f"{TEXT_DIR}/{src}_{split}.txt", f"{BIN_DIR}/{src}_{split}.bin"
    n_tok = n_doc = 0; t0 = time.time()
    with open(inp, encoding="utf-8") as fi, open(out, "wb") as fo:
        batch = []
        def flush(batch):
            encs = tok.encode_batch(batch, add_special_tokens=False)
            parts = []
            for e in encs:
                parts.append(np.asarray(e.ids, dtype=np.uint16)); parts.append(np.array([EOS_ID], dtype=np.uint16))
            arr = np.concatenate(parts); arr.tofile(fo)
            return len(arr)
        for line in fi:
            batch.append(line.rstrip("\n"))
            if len(batch) == batch_size:
                n_tok += flush(batch); n_doc += len(batch); batch = []
                if n_doc % (batch_size * 200) == 0:
                    print(f"   {src}_{split}: {n_doc:,} docs, {n_tok/1e6:,.0f}M tokens, "
                          f"{n_tok/(time.time()-t0)/1e6:.2f}M tok/s")
        if batch:
            n_tok += flush(batch); n_doc += len(batch)
    assert os.path.getsize(out) == 2 * n_tok
    print(f"{src}_{split:5s}: {n_doc:>9,} docs -> {n_tok/1e6:9.2f}M tokens  ({time.time()-t0:.0f}s)")
    return n_tok

TOKEN_COUNTS = {}
for src in ["supreme_court", "acts", "tribunals", "high_courts"]:
    for split in ["val", "train"]:
        TOKEN_COUNTS[f"{src}_{split}"] = tokenize_file(src, split)
print(f"\nTOTAL train tokens: {sum(v for k, v in TOKEN_COUNTS.items() if k.endswith('_train'))/1e9:.3f} B")

supreme_court_val  :       555 docs ->      2.49M tokens  (5s)
supreme_court_train:    54,333 docs ->    250.74M tokens  (441s)
acts_val  :       208 docs ->      1.74M tokens  (3s)
acts_train:    18,609 docs ->    148.80M tokens  (281s)
tribunals_val  :     2,477 docs ->      4.32M tokens  (7s)
   tribunals_train: 102,400 docs, 152M tokens, 0.59M tok/s
   tribunals_train: 204,800 docs, 315M tokens, 0.62M tok/s
tribunals_train:   294,980 docs ->    466.99M tokens  (756s)
high_courts_val  :     4,125 docs ->      4.25M tokens  (8s)
   high_courts_train: 102,400 docs, 57M tokens, 0.55M tok/s
   high_courts_train: 204,800 docs, 216M tokens, 0.58M tok/s
   high_courts_train: 307,200 docs, 294M tokens, 0.59M tok/s
   high_courts_train: 409,600 docs, 351M tokens, 0.59M tok/s
high_courts_train:   510,960 docs ->    454.55M tokens  (769s)

TOTAL train tokens: 1.321 B


In [12]:
# ===================== CELL 8 - Sanity-check the .bin files and upload everything to our private HF repo =====================
rng = np.random.default_rng(0)
for name in TOKEN_COUNTS:
    if TOKEN_COUNTS[name] == 0:
        print(f"{name:22s} EMPTY (no documents)"); continue
    arr = np.memmap(f"{BIN_DIR}/{name}.bin", dtype=np.uint16, mode="r")
    assert len(arr) == TOKEN_COUNTS[name] and int(arr.max()) < VOCAB_SIZE
    i = int(rng.integers(0, max(1, len(arr) - 60)))
    print(f"{name:22s} {len(arr)/1e6:9.2f}M tokens | eos count {int((arr == EOS_ID).sum()):,} | "
          f"sample: {tok.decode(arr[i:i+40].tolist())[:110]!r}")

meta = {
    "vocab_size": VOCAB_SIZE, "eos_id": EOS_ID, "pad_id": tok.token_to_id("<|pad|>"), "specials": SPECIALS,
    "dtype": "uint16", "token_counts": TOKEN_COUNTS, "normalisation": "NFKC + ASCII punctuation map + drop non-ASCII + "
    "collapse whitespace (matches test prompts)", "tokenizer": "byte-level BPE, add_prefix_space=False",
    "tokens_per_word_test": {"ours": ours, "gpt2": gpt2}, "n_dev": len(dev_prompts),
    "prep": json.load(open(f"{OUT_DIR}/prep_stats.json")),
}
with open(f"{OUT_DIR}/meta.json", "w") as f:
    json.dump(meta, f, indent=2)

t0 = time.time()
api.upload_folder(repo_id=DATA_REPO, repo_type="dataset", folder_path=OUT_DIR,
                  allow_patterns=["bin/*.bin", "tokenizer.json", "dev.json", "dev_truth.json", "meta.json",
                                  "prep_stats.json"],
                  commit_message="Phase-1 prepared data: tokenizer + tokenized corpus + dev set")
print(f"Uploaded to https://huggingface.co/datasets/{DATA_REPO} in {time.time()-t0:.0f}s")
print(sorted(f for f in api.list_repo_files(DATA_REPO, repo_type="dataset")))

supreme_court_val           2.49M tokens | eos count 555 | sample: '879 to be beyond the power of the Legislature." In Syed Mohamed & Co. v. The State of Madras(2), the question '
supreme_court_train       250.74M tokens | eos count 54,333 | sample: ' definitely a new product is prepared or made, which is known as a different item and the said item cannot be '
acts_val                    1.74M tokens | eos count 208 | sample: ' its use for certain purposes is notified as a mineral other than a minor mineral and in relation to its use f'
acts_train                148.80M tokens | eos count 18,609 | sample: ' u | 20% 8517 79 90 | - - - Other | u | 15% (xi) in heading 8519, tariff item 8519 50 00 and the entries relat'
tribunals_val               4.32M tokens | eos count 2,477 | sample: ' the PIO providing the required information upto 26/05/1996 (therefore, incomplete). Grounds for Second Appeal'
tribunals_train           466.99M tokens | eos count 294,980 | sample: ' it has been provide

No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded to https://huggingface.co/datasets/Abhayk24/cs781-p1-data in 12s
['.gitattributes', 'bin/acts_train.bin', 'bin/acts_val.bin', 'bin/high_courts_train.bin', 'bin/high_courts_val.bin', 'bin/supreme_court_train.bin', 'bin/supreme_court_val.bin', 'bin/tribunals_train.bin', 'bin/tribunals_val.bin', 'dev.json', 'dev_truth.json', 'meta.json', 'prep_stats.json', 'tokenizer.json']
